# 🚦 Traffic Sign Classification (PyTorch) - Training Notebook

Huấn luyện **hai mô hình CNN** phân loại crop biển báo (do YOLOv8 phát hiện), rồi so sánh và ensemble:
- **Custom CNN** (VGG-style, baseline) - `Utils/models.py::CustomCNN`
- **ResNet-18** (tự cài đặt, train từ đầu, chỉnh cho ảnh 64x64) - `Utils/models.py::ResNet18`
- **Ensemble**: trung bình xác suất của hai mô hình

**Input:** ảnh RGB 64x64, giá trị [0, 1]. Model lưu thành `.pt`, copy vào `Inference/`.

**Dataset:** tạo bằng `Data_Pipeline/build_cnn_crops.py` (thư mục `train/val/test/<id>_<tên>/`). Đặt đường dẫn bằng biến môi trường `DATASET_PATH` hoặc sửa ở cell cấu hình.

In [1]:
import os, sys, copy
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.metrics import classification_report, confusion_matrix

# Import Utils/models.py (local: ../Utils, Kaggle: models.py cạnh notebook)
sys.path.insert(0, str(Path.cwd().parent))
sys.path.insert(0, str(Path.cwd()))
try:
    from Utils.models import build_model, save_checkpoint
except ImportError:
    from models import build_model, save_checkpoint

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("PyTorch:", torch.__version__, "| Device:", DEVICE)
if DEVICE.type == 'cpu':
    print("⚠️ Không có GPU. Hãy bật Accelerator trong Session Options!")

PyTorch: 2.6.0+cu124 | Device: cuda


## 2. Cấu hình

In [2]:
DATASET_PATH = os.environ.get('DATASET_PATH', str(Path.cwd().parent.parent / 'Data_Processed' / 'cnn_dataset'))
TRAIN_DIR = os.path.join(DATASET_PATH, 'train')
VAL_DIR = os.path.join(DATASET_PATH, 'val')
TEST_DIR = os.path.join(DATASET_PATH, 'test')

IMG_SIZE = 64
BATCH_SIZE = 128
EPOCHS = 30
NUM_WORKERS = 4
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

## 3. Dữ liệu & Augmentation
- **Train:** xoay ±10°, zoom nhẹ.
- **Val/Test:** chỉ resize + ToTensor ([0, 1]).

In [3]:
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomAffine(degrees=10, scale=(0.9, 1.1)),
    transforms.ToTensor(),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

train_ds = datasets.ImageFolder(TRAIN_DIR, train_tf)
val_ds = datasets.ImageFolder(VAL_DIR, eval_tf)
test_ds = datasets.ImageFolder(TEST_DIR, eval_tf)

class_names = train_ds.classes
NUM_CLASSES = len(class_names)
print(f"✅ {NUM_CLASSES} lớp: {train_ds.class_to_idx}")
print(f"Train/Val/Test: {len(train_ds)}/{len(val_ds)}/{len(test_ds)}")

pin = DEVICE.type == 'cuda'
train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=pin)
val_loader = DataLoader(val_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)
test_loader = DataLoader(test_ds, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=pin)

# Trọng số lớp (nghịch đảo căn bậc hai tần suất) để giảm lệch dữ liệu
counts = np.bincount(train_ds.targets, minlength=NUM_CLASSES)
class_weights = torch.tensor((counts.sum() / counts) ** 0.5, dtype=torch.float32)
class_weights = (class_weights / class_weights.mean()).to(DEVICE)
print("Số mẫu mỗi lớp:", dict(zip(class_names, counts)))

✅ 5 lớp: {'0_P': 0, '1_W': 1, '2_R': 2, '3_I': 3, '5_S': 4}
Train/Val/Test: 29317/3974/2259
Số mẫu mỗi lớp: {'0_P': np.int64(14886), '1_W': np.int64(9564), '2_R': np.int64(2716), '3_I': np.int64(1514), '5_S': np.int64(637)}


## 4. Hàm huấn luyện
- Lưu checkpoint theo `val_acc` tốt nhất, `EarlyStopping` theo `val_loss`, `ReduceLROnPlateau`.

In [4]:
scaler = torch.amp.GradScaler(enabled=DEVICE.type == 'cuda')


def run_epoch(model, loader, criterion, optimizer=None):
    train = optimizer is not None
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == 'cuda'):
                out = model(x)
                loss = criterion(out.float(), y)
            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * y.size(0)
            correct += (out.argmax(1) == y).sum().item()
            n += y.size(0)
    return total_loss / n, correct / n


def fit(model, kind, ckpt_path, epochs, lr, patience=10):
    criterion = nn.CrossEntropyLoss(weight=class_weights)  # bù lớp ít mẫu
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=3, min_lr=1e-7)
    hist = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
    best_acc, best_loss, bad = 0.0, float('inf'), 0
    for ep in range(1, epochs + 1):
        tl, ta = run_epoch(model, train_loader, criterion, optimizer)
        vl, va = run_epoch(model, val_loader, criterion)
        scheduler.step(vl)
        for k, v in zip(hist, (tl, ta, vl, va)):
            hist[k].append(v)
        print(f"Epoch {ep:02d}/{epochs} | loss {tl:.4f} acc {ta:.4f} | val_loss {vl:.4f} val_acc {va:.4f} | lr {optimizer.param_groups[0]['lr']:.2e}")
        if va > best_acc:
            best_acc = va
            save_checkpoint(model, kind, class_names, ckpt_path)
        if vl < best_loss:
            best_loss, bad = vl, 0
        else:
            bad += 1
            if bad >= patience:
                print("⏹️ Early stopping")
                break
    # Nạp lại checkpoint tốt nhất
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE)['state_dict'])
    print(f"✅ Best val_acc: {best_acc:.4f} -> {ckpt_path}")
    return hist

## 5. Custom CNN

In [5]:
CNN_PATH = 'best_cnn_traffic_sign.pt'
cnn = build_model('cnn', NUM_CLASSES).to(DEVICE)
print(f"Tham số: {sum(p.numel() for p in cnn.parameters()):,}")
history_cnn = fit(cnn, 'cnn', CNN_PATH, EPOCHS, lr=1e-3, patience=10)

Tham số: 2,494,469


Epoch 01/30 | loss 0.5275 acc 0.8656 | val_loss 0.6255 val_acc 0.8546 | lr 1.00e-03


Epoch 02/30 | loss 0.3234 acc 0.9202 | val_loss 0.1042 val_acc 0.9683 | lr 1.00e-03


Epoch 03/30 | loss 0.2787 acc 0.9325 | val_loss 0.1048 val_acc 0.9731 | lr 1.00e-03


Epoch 04/30 | loss 0.2494 acc 0.9401 | val_loss 0.1354 val_acc 0.9620 | lr 1.00e-03


...

Epoch 22/30 | loss 0.0359 acc 0.9914 | val_loss 0.0526 val_acc 0.9884 | lr 2.50e-04


Epoch 23/30 | loss 0.0275 acc 0.9930 | val_loss 0.0522 val_acc 0.9884 | lr 2.50e-04


Epoch 24/30 | loss 0.0319 acc 0.9924 | val_loss 0.0501 val_acc 0.9874 | lr 1.25e-04


Epoch 25/30 | loss 0.0269 acc 0.9933 | val_loss 0.0450 val_acc 0.9897 | lr 1.25e-04


Epoch 26/30 | loss 0.0272 acc 0.9931 | val_loss 0.0433 val_acc 0.9907 | lr 1.25e-04
⏹️ Early stopping
✅ Best val_acc: 0.9907 -> best_cnn_traffic_sign.pt


## 6. ResNet-18 (tự cài đặt)
Train từ đầu (không pretrained) với LR 1e-3, Adam + giảm LR theo val loss.

In [6]:
RES_PATH = 'best_resnet18_traffic_sign.pt'
res = build_model('resnet18', NUM_CLASSES).to(DEVICE)
print(f"Tham số: {sum(p.numel() for p in res.parameters()):,}")
history_res = fit(res, 'resnet18', RES_PATH, EPOCHS, lr=1e-3, patience=8)

Tham số: 11,179,077


Epoch 01/30 | loss 0.5845 acc 0.8525 | val_loss 0.2076 val_acc 0.9416 | lr 1.00e-03


Epoch 02/30 | loss 0.3465 acc 0.9164 | val_loss 0.1118 val_acc 0.9690 | lr 1.00e-03


Epoch 03/30 | loss 0.2225 acc 0.9453 | val_loss 0.1348 val_acc 0.9602 | lr 1.00e-03


Epoch 04/30 | loss 0.2179 acc 0.9476 | val_loss 0.1432 val_acc 0.9577 | lr 1.00e-03


...

Epoch 17/30 | loss 0.0198 acc 0.9950 | val_loss 0.0458 val_acc 0.9897 | lr 2.50e-04


Epoch 18/30 | loss 0.0120 acc 0.9971 | val_loss 0.0444 val_acc 0.9907 | lr 2.50e-04


Epoch 19/30 | loss 0.0118 acc 0.9975 | val_loss 0.0540 val_acc 0.9889 | lr 2.50e-04


Epoch 20/30 | loss 0.0079 acc 0.9984 | val_loss 0.0484 val_acc 0.9909 | lr 1.25e-04


⏹️ Early stopping
✅ Best val_acc: 0.9909 -> best_resnet18_traffic_sign.pt


## 7. Đánh giá & Trực quan hóa

In [7]:
def plot_history(hist, title):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(hist['accuracy'], label='Train'); plt.plot(hist['val_accuracy'], label='Val')
    plt.title(f'{title} - Accuracy'); plt.xlabel('Epoch'); plt.legend(); plt.grid(True)
    plt.subplot(1, 2, 2)
    plt.plot(hist['loss'], label='Train'); plt.plot(hist['val_loss'], label='Val')
    plt.title(f'{title} - Loss'); plt.xlabel('Epoch'); plt.legend(); plt.grid(True)
    plt.show()

def report(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - {title}'); plt.ylabel('True'); plt.xlabel('Predicted')
    plt.show()
    print(classification_report(y_true, y_pred, target_names=class_names))

@torch.no_grad()
def predict_probs(model, loader):
    model.eval()
    return np.concatenate([torch.softmax(model(x.to(DEVICE)), 1).cpu().numpy() for x, _ in loader])

plot_history(history_cnn, 'Custom CNN')
plot_history(history_res, 'ResNet-18')

y_true = np.array(test_ds.targets)  # test_loader không shuffle nên thứ tự khớp
probs = {'Custom CNN': predict_probs(cnn, test_loader),
         'ResNet-18': predict_probs(res, test_loader)}
probs['Ensemble (CNN + ResNet-18)'] = (probs['Custom CNN'] + probs['ResNet-18']) / 2

for name, p in probs.items():
    y_pred = p.argmax(1)
    print(f"\n===== {name} | Test Accuracy: {(y_pred == y_true).mean() * 100:.2f}% =====")
    report(y_true, y_pred, name)

/tmp/ipykernel_53040/1493635988.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()



===== Custom CNN | Test Accuracy: 99.78% =====
              precision    recall  f1-score   support

         0_P       1.00      1.00      1.00      1115
         1_W       1.00      1.00      1.00       810
         2_R       0.99      0.99      0.99       199
         3_I       0.99      0.98      0.98        96
         5_S       0.97      1.00      0.99        39

    accuracy                           1.00      2259
   macro avg       0.99      0.99      0.99      2259
weighted avg       1.00      1.00      1.00      2259


===== ResNet-18 | Test Accuracy: 99.73% =====
              precision    recall  f1-score   support

         0_P       1.00      1.00      1.00      1115
         1_W       1.00      1.00      1.00       810
         2_R       0.99      1.00      1.00       199
         3_I       0.98      0.98      0.98        96
         5_S       0.95      1.00      0.97        39

    accuracy                           1.00      2259
   macro avg       0.98      1.00   

/tmp/ipykernel_53040/1493635988.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_53040/1493635988.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/tmp/ipykernel_53040/1493635988.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [8]:
# Tải model về máy (nếu chạy trên Kaggle)
from IPython.display import FileLink, display
display(FileLink(CNN_PATH))
display(FileLink(RES_PATH))

/home/uythuat/Desktop/vietnam-traffic-sign-recognition-main/Training/best_cnn_traffic_sign.pt

/home/uythuat/Desktop/vietnam-traffic-sign-recognition-main/Training/best_resnet18_traffic_sign.pt